<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-HUYTRUONG/AIO/blob/aio/M4-Foundation-DL/Python-based-DS/M04W01%20-%20Data%20Cleaning%26Transformation%20Using%20Pandas/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -q \
    "https://raw.githubusercontent.com/AIVIETNAM-AIO-HUYTRUONG/AIO/refs/heads/aio/helpers/dataset_helper.py" \
    -O /content/helper.py

%run /content/helper.py

In [4]:
# https://raw.githubusercontent.com/AIVIETNAM-AIO-HUYTRUONG/AIO/refs/heads/aio/M4-Foundation-DL/Python-based-DS/Data/data_cleaning_don_hang_raw.csv
CSV_URL = input("Paste CSV URL: ").strip()

df = load_csv_from_url(
    url=CSV_URL,
    data_dir="/content/data",
    filename="data_cleaning_don_hang_raw.csv"
)

df.head()

Paste CSV URL: https://raw.githubusercontent.com/AIVIETNAM-AIO-HUYTRUONG/AIO/refs/heads/aio/M4-Foundation-DL/Python-based-DS/Data/data_cleaning_don_hang_raw.csv
https://raw.githubusercontent.com/AIVIETNAM-AIO-HUYTRUONG/AIO/refs/heads/aio/M4-Foundation-DL/Python-based-DS/Data/data_cleaning_don_hang_raw.csv
Dataset downloaded successfully:
/content/data/data_cleaning_don_hang_raw.csv
Dataset loaded successfully:
- Path: /content/data/data_cleaning_don_hang_raw.csv
- Rows: 6
- Columns: 5


,OrderID,Customer,City,Quantity,OrderDate
0,O101,An,HCM,2,2026-09-01
1,O102,Binh,HN,NaN,2026-09-02
2,O103,Chi,hcm,two,2026-09-03
3,O104,Dung,Da Nang,3,2026-09-04
4,O104,Dung,Da Nang,3,2026-09-04


# Common Data Quality Issues
- Đọc dữ liệu thô: thiếu giá trị, trùng dòng, sai kiểu dữ liệu, text không nhất quán.

In [5]:
df = pd.read_csv("/content/data/data_cleaning_don_hang_raw.csv")
df

,OrderID,Customer,City,Quantity,OrderDate
0,O101,An,HCM,2,2026-09-01
1,O102,Binh,HN,NaN,2026-09-02
2,O103,Chi,hcm,two,2026-09-03
3,O104,Dung,Da Nang,3,2026-09-04
4,O104,Dung,Da Nang,3,2026-09-04
5,O105,Em,Ho Chi Minh,1,2026-09-05


## Detect Missing Values
- `isna()` phát hiện giá trị thiếu, `.sum()` đếm số giá trị thiếu theo cột.

In [10]:
df.isna()

,OrderID,Customer,City,Quantity,OrderDate
0,False,False,False,False,False
1,False,False,False,True,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
5,False,False,False,False,False


In [11]:
df.isna().sum()

,0
OrderID,0
Customer,0
City,0
Quantity,1
OrderDate,0


In [15]:
mask = df["Quantity"].isna()
df.loc[mask]

,OrderID,Customer,City,Quantity,OrderDate
1,O102,Binh,HN,NaN,2026-09-02


# When to Remove Missing Values
- `dropna(subset=[...])` xoá dòng bị thiếu giá trị bắt buộc.

In [16]:
df_removed = df.dropna(subset = ["Quantity"])
df_removed

,OrderID,Customer,City,Quantity,OrderDate
0,O101,An,HCM,2,2026-09-01
2,O103,Chi,hcm,two,2026-09-03
3,O104,Dung,Da Nang,3,2026-09-04
4,O104,Dung,Da Nang,3,2026-09-04
5,O105,Em,Ho Chi Minh,1,2026-09-05


In [17]:
df_removed.isna().sum()

,0
OrderID,0
Customer,0
City,0
Quantity,0
OrderDate,0


# Choose How To Fill Mising Values
- `fillna()` điền giá trị thiếu bằng một quy tắc nghiệp vụ có căn cứ.

In [22]:
df_filled = df.copy()
df_filled['Quantity'] = df_filled['Quantity'].fillna(1)
df_filled['Quantity'].isna().sum()

np.int64(0)

# Detect and Resolve Duplicate Rows
- `duplicated(keep=False)` đánh dấu toàn bộ các dòng trùng. `drop_duplicates()` chỉ giữ lại bản ghi đầu tiên.

In [25]:
mask = df_filled.duplicated(keep=False)
df.loc[mask]

,OrderID,Customer,City,Quantity,OrderDate
3,O104,Dung,Da Nang,3,2026-09-04
4,O104,Dung,Da Nang,3,2026-09-04


In [26]:
df_debup = df_filled.drop_duplicates(keep="first")
df_debup

,OrderID,Customer,City,Quantity,OrderDate
0,O101,An,HCM,2,2026-09-01
1,O102,Binh,HN,1,2026-09-02
2,O103,Chi,hcm,two,2026-09-03
3,O104,Dung,Da Nang,3,2026-09-04
5,O105,Em,Ho Chi Minh,1,2026-09-05


# Incorrect Data Types
- So sánh kiểu dữ liệu đang lưu với ý nghĩa mong muốn của từng cột.

In [27]:
df_debup.dtypes

,0
OrderID,object
Customer,object
City,object
Quantity,object
OrderDate,object


# Convert Numeric and Date Values

- `pd.to_numeric()` và `pd.to_datetime()` với `errors="coerce"` chuyển giá trị bất thường thành NaN/NaT để rà soát.

In [50]:
df_typed = df_debup.copy()
df_typed['Quantity'] = pd.to_numeric(
    df_typed['Quantity'].replace({"two":"2"}),
    errors="coerce"
)

df_typed["OrderDate"] = pd.to_datetime(
    df_typed["OrderDate"],
    format="%Y-%m-%d",
    errors="coerce"
)

df_typed

,OrderID,Customer,City,Quantity,OrderDate
0,O101,An,HCM,2,2026-09-01
1,O102,Binh,HN,1,2026-09-02
2,O103,Chi,hcm,2,2026-09-03
3,O104,Dung,Da Nang,3,2026-09-04
5,O105,Em,Ho Chi Minh,1,2026-09-05


# Standardize Inconsistent Text Values
- Nhiều nhãn cùng ý nghĩa được gộp về một nhãn chuẩn (canonical label).

In [39]:
df_typed["City"].unique()

array(['HCM', 'HN', 'hcm', 'Da Nang', 'Ho Chi Minh'], dtype=object)

In [52]:
city_map = {
    "hcm":"HCM",
    "Ho Chi Minh":"HCM"
}

df_text = df_typed.copy()
df_text['City'] = df_text["City"].replace(city_map)
df_text["City"].value_counts()

,count
City,
HCM,3
HN,1
Da Nang,1


# Validate the Cleaned Data
- Kiểm tra lại dữ liệu trước khi dùng cho phân tích hoặc transform.

In [55]:
df_text_dup = df_text.set_index("OrderID")
df_text_dup.isna().sum()

,0
Customer,0
City,0
Quantity,0
OrderDate,0


In [56]:
df_text_dup.duplicated().sum()

np.int64(0)

In [57]:
df_text.index.is_unique

True

In [58]:
df_text[["Quantity", "OrderDate"]].dtypes

,0
Quantity,int64
OrderDate,datetime64[ns]


In [59]:
sorted(df_text["City"].unique())

['Da Nang', 'HCM', 'HN']

In [60]:
df_text.shape

(5, 5)

In [61]:
df_clean = df_text.copy()
df_clean

,OrderID,Customer,City,Quantity,OrderDate
0,O101,An,HCM,2,2026-09-01
1,O102,Binh,HN,1,2026-09-02
2,O103,Chi,HCM,2,2026-09-03
3,O104,Dung,Da Nang,3,2026-09-04
5,O105,Em,HCM,1,2026-09-05
